# SOXX/QQQ 下跌 beta 缺口与原 Stage1 五日尾部标签：必要信息检验

固定正向尾部风险 AUC、完整日历和一次区块重抽；历史来源时点未认证，不据此建账户。


In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from sklearn.metrics import roc_auc_score
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_soxx_qqq_downside_beta_tail_h5_necessary_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('external_requests','new_model_fits','new_native_accounts'))
a=root/src['feature_card'];b=root/src['label_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['feature_summary_sha256'] and sha(b/'summary.json')==src['label_summary_sha256']
assert sa['recorder_id']==src['feature_recorder_id'] and sb['recorder_id']==src['label_recorder_id']
assert sa['nas_evidence']['sha256']==src['feature_NAS_sha256'] and sb['nas_evidence']['sha256']==src['label_NAS_sha256']
assert sha(a/'qlib_recorder'/src['feature_experiment_id']/src['feature_recorder_id']/'artifacts/analysis/source_frame')==src['feature_frame_sha256']
assert sha(b/'qlib_recorder'/src['label_experiment_id']/src['label_recorder_id']/'artifacts/analysis/daily')==src['label_daily_sha256']
assert sha(root/src['Qlib_provider']/'qlib.yaml')==src['Qlib_config_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
with R.uri_context(sa['recorder_uri']):feature=R.get_recorder(recorder_id=src['feature_recorder_id'],experiment_id=src['feature_experiment_id']).load_object('analysis/source_frame')
with R.uri_context(sb['recorder_uri']):label=R.get_recorder(recorder_id=src['label_recorder_id'],experiment_id=src['label_experiment_id']).load_object('analysis/daily')
cal=pd.DatetimeIndex(D.calendar(start_time=plan['cohort']['start'],end_time=plan['cohort']['end'],freq='day'));assert len(cal)==plan['cohort']['full_calendar_rows']==len(label)==944
assert cal.equals(pd.DatetimeIndex(pd.to_datetime(label.signal_date)))
score=feature.downside_beta_gap_60.reindex(cal);assert score.notna().all() and np.isfinite(score).all()
mature=label.tail_h5.notna() & label.label_available_h5.notna();assert mature.sum()==plan['cohort']['mature_H5_rows']==940 and label.index[~mature].tolist()==[940,941,942,943]
assert (pd.to_datetime(label.loc[mature,'label_available_h5'])>pd.to_datetime(label.loc[mature,'trade_date'])).all()
d=pd.DataFrame({'signal_date':cal,'trade_date':pd.to_datetime(label.trade_date),'downside_beta_gap_60':score.to_numpy(dtype=float),'tail_h5':label.tail_h5.to_numpy(dtype=float),'H5_mature':mature.to_numpy(dtype=bool),'H5_available_at':label.label_available_h5,'saved_Stage1_tail_probability':label.frozen_current_tail_probability.to_numpy(dtype=float)})
assert d.loc[d.H5_mature,'tail_h5'].sum()==177
static=pd.concat({'feature':d[['downside_beta_gap_60']],'label':d[['tail_h5']],'meta':d[['H5_mature']]},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['SOXX_QQQ']*len(cal)],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':(plan['cohort']['start'],plan['cohort']['end'])});assert len(ds.prepare('held',col_set=['feature','label','meta']))==944
auc=lambda z,col='downside_beta_gap_60':float(roc_auc_score(z.tail_h5,z[col])) if len(z)>10 and z.tail_h5.nunique()==2 else None
h=d.loc[d.H5_mature];primary=auc(h);saved=auc(h,'saved_Stage1_tail_probability');assert primary is not None and saved is not None
half1=h.loc[h.signal_date<pd.Timestamp('2025-01-01')];half2=h.loc[h.signal_date>=pd.Timestamp('2025-01-01')];halves=[auc(half1),auc(half2)];assert all(x is not None for x in halves)
annual={str(y):{'rows':len(g),'positive_tail':int(g.tail_h5.sum()),'AUC':auc(g),'mean_beta_gap':float(g.downside_beta_gap_60.mean())} for y,g in h.groupby(h.signal_date.dt.year)}
quarters={str(q):{'rows':len(g),'positive_tail':int(g.tail_h5.sum()),'AUC':auc(g),'mean_beta_gap':float(g.downside_beta_gap_60.mean())} for q,g in h.groupby(h.signal_date.dt.to_period('Q'))};assert len(quarters)==16
cfg=plan['statistics']['uncertainty'];tracking=card/'qlib_recorder';state={'plan_sha256':sha(card/'plan.json'),'bootstrap_attempted':True,'bootstrap_draw_budget':cfg['draws'],'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rng=np.random.default_rng(cfg['seed']);n=len(d);rep=np.empty(cfg['draws'])
for i in range(cfg['draws']):
 idx=np.empty(n,dtype=np.int32);idx[0]=rng.integers(n)
 for j in range(1,n):idx[j]=rng.integers(n) if rng.random()<1/cfg['mean_block_sessions'] else (idx[j-1]+1)%n
 sample=d.iloc[idx];sample=sample.loc[sample.H5_mature & sample.tail_h5.notna()];rep[i]=auc(sample)
assert np.isfinite(rep).all();ci=np.quantile(rep,cfg['two_sided_quantiles']);support=bool(primary>0.5 and all(x>0.5 for x in halves) and all(annual[str(y)]['AUC'] is not None and annual[str(y)]['AUC']>0.5 for y in plan['cohort']['complete_years']) and ci[0]>0.5)
report={'decision':'conditional_SOXX_QQQ_downside_beta_tail_H5_necessary_support_development_only' if support else 'closed_fixed_SOXX_QQQ_downside_beta_tail_H5_necessary_information_failed','primary_H5_AUC':primary,'saved_Stage1_tail_probability_AUC_descriptive':saved,'held_halves_H5_AUC':halves,'complete_2024_2025_H5_AUC':{str(y):annual[str(y)]['AUC'] for y in (2024,2025)},'descriptive_95pct_full_calendar_block_interval':[float(ci[0]),float(ci[1])],'necessary_support':support,'annual':annual,'quarters':quarters,'calendar_rows':944,'mature_H5_rows':940,'tail_positive_rows':177,'bootstrap_replicates':len(rep),'new_model_fits':0,'new_native_accounts':0,'historical_independence':False,'historical_PIT_qualified':False,'project_wide_search_corrected':False}
assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds);rec.save_objects(artifact_path='analysis',matched_daily=d,report=report,bootstrap_replicates=rep);rec.log_metrics(primary_H5_AUC=primary,block_lower=float(ci[0]),necessary_support=int(support),model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Current revised Yahoo source/first receipts unqualified; development-only conditional screening.','Reused 2023-2026 history and 2000-draw interval do not create project-wide independent validation.','Marginal AUC is not incremental original39 tail-head information, Stage1 fee-net account or SOXX success.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='downbeta-tail-h5-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'AUC':primary,'old_AUC':saved,'halves':halves,'complete_years':report['complete_2024_2025_H5_AUC'],'ci':report['descriptive_95pct_full_calendar_block_interval'],'support':support,'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[26026:MainThread](2026-10-23 08:02:30,056) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[26026:MainThread](2026-10-23 08:02:30,058) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[26026:MainThread](2026-10-23 08:02:30,058) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[26026:MainThread](2026-10-23 08:02:32,085) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_soxx_qqq_downside_beta_tail_h5_necessary_information_20261023.


[26026:MainThread](2026-10-23 08:02:32,086) INFO - qlib.workflow - [exp.py:258] - Experiment 526166212591487693 starts running ...


[26026:MainThread](2026-10-23 08:02:32,141) INFO - qlib.workflow - [recorder.py:345] - Recorder b6bc90a4254142d48069f7a0fd442aea starts running under Experiment 526166212591487693 ...


[26026:MainThread](2026-10-23 08:02:33,236) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "AUC": 0.5570562232045672,
  "old_AUC": 0.5709768902118459,
  "halves": [
    0.504265873015873,
    0.5619052997148262
  ],
  "complete_years": {
    "2024": 0.4628956927867151,
    "2025": 0.6202
  },
  "ci": [
    0.4849974133821186,
    0.6254952055144989
  ],
  "support": false,
  "recorder_id": "b6bc90a4254142d48069f7a0fd442aea"
}


# SOXX/QQQ 下跌 beta 缺口与五日尾部风险：必要信息检验

冻结前 60 个完整 Qlib 交易日的 `SOXX 对 QQQ 下跌日 beta − 全部日 beta`，只检查它能否按预设**正向**排序原 Stage1 的五日 SOXX 尾部事件。944 个完整复用评价日全部保留；940 个成熟标签中 177 个尾部事件，末尾 4 日未知。来源使用当前修订的 Yahoo 历史，历史首次收取时点仍未认证。

全期 AUC **0.55706**，但固定前后半段为 **0.50427 / 0.56191**；完整 2024、2025 年为 **0.46290 / 0.62020**，2026 不完整年度为 **0.4233**。原 Stage1 保存的尾部概率在相同 940 日上的描述性 AUC 为 **0.57098**。按完整 944 日历、平均区块 63 日、2,000 次 stationary bootstrap 的描述性 95% 区间 **[0.48500, 0.62550]**，下界没有超过随机排序的 0.5。全部四年和 16 季度留在 Recorder；不能因 2025 较好而选择年份或改变公式。

原生 Qlib Recorder `b6bc90a4254142d48069f7a0fd442aea` 已读回 944 行 DatasetH、940 个成熟标签、177 个尾部事件与 2,000 个重抽样；NAS 哈希冷读通过。0 新拟合、0 账户。这个边际 AUC 也不是在原 39 项和既有尾部输出之上的增量信息，更不是费用后 SOXX 超额。

**决定：`closed_fixed_SOXX_QQQ_downside_beta_tail_H5_necessary_information_failed`，不推广。** 此固定正向、60 日、五日尾部版本关闭；不按 2025 结果反向调窗、选 QQQ 跌幅阈值或风险状态。下一机制需要不同的决策前信息，并先证明时点与增量，再在完整 Stage1 0bp Tiered 自有账户上接受 SOXX 硬门。
